# Efficient Multimodal Reasoning: final-report GPU runs (Ideas 1, 3, 4)

> **Main path: AWS.** Colab could not allocate an L4 on Nov 23, so the runs moved to an AWS g6.2xlarge (the same L4 GPU) with `python -m src.final.run_all`, which runs these same sections with the same config lists (README, "Final report runs on AWS"). This notebook is the Colab fallback.

**Team 7, 11-777 Fall 2026.** Every model pass for the final report, on **one GPU type (Colab L4, bf16)**, so accuracy, tokens and wall-clock time are comparable across ideas.
Model: Qwen2.5-Omni-3B (frozen). Data: OmniBench, all 1,141 usable items, in the frozen 5 folds of `data/final_folds.json`.
Every accuracy is averaged over 3 option orders, as in the midterm.

| § | Runs | For | Owner (reviews) | Passes | L4 time (est.) |
|---|---|---|---|---|---|
| 6 | 7 base input formats × 1,141 items | Ideas 1, 3, 4 (baselines) | Arjun, Lindy | 23,961 | ~1.6 h |
| 7 | Cheap probe cells `(64,½)`, `(64,0)`, `(0,½)` × 1,141 | Idea 1 features, Idea 3 first pass | Arjun, Lindy, Alvin | 10,269 | ~0.7 h |
| 8 | Generate the Idea 4 text for 242 items | Idea 4 | Nikkita | – | ~55 min |
| 9 | Score 11 Idea 4 configs × 242 | Idea 4 | Nikkita | 7,986 | ~35 min |
| 9b | Generation-length ablation: 6 Idea 4 configs at 80 and 120 tokens × 242 | Idea 4 | Nikkita | 8,712 | ~30 min |
| 9c | Speech-threshold sweep inputs: 4 routing branches on every item × 242 | Idea 4 | Nikkita | 2,904 | ~12 min |
| 10 | The other 9 budget-grid cells × 1,141 | Idea 3 | Alvin | 30,807 | ~2 h |
| 11 | Audio selection: centered and loudest window × 1,141 | Idea 3 | Alvin | 20,538 | ~1 h |
| | | | | **105,177** | **≈ 7 h** |

§6–9 give every idea its core data in about 3.5 h; §9b and §9c are the Idea 4 generation-length ablation and speech-threshold sweep; §10–11 finish Idea 3. No model is trained here: the Idea 1 router and the Idea 3 allocator run on CPU afterwards, from these outputs (`src/final/bank.py`).

**Who runs it:** reviewed by the whole team, then run by Lindy on behalf of the team. Outputs land in the runner's `MyDrive/mmml_final/`: share that folder with the team (right-click → Share).

**How to run:**
1. Runtime → Change runtime type → **L4 GPU**.
2. Run §0–5, then §6 onwards. **Keep the tab open**: Colab Pro has no background execution.
3. **If it disconnects:** reconnect, run §0–4, then rerun the section you were on. Finished passes are skipped.

**What survives a disconnect** (all in `MyDrive/mmml_final/`):
- results are written straight to Drive;
- logs go to `logs/`;
- timestamped copies of every results file go to `backups/` at the start of each session and every 15 min (the last 3 are kept);
- a line cut off mid-write is skipped and that pass is re-run.

## 0. Run settings: review these as a team

Everything that defines the experiments is listed here (the rest lives in `configs/default.yaml` and the scripts named below). **Decisions to confirm in the review:**

| Setting | Value | Where | Owner |
|---|---|---|---|
| Folds | 5, stratified by audio type, seed 2026 (frozen) | `data/final_folds.json` | all |
| Option orders per item | 3 (original + 2 seeded shuffles) | `configs/default.yaml` | all |
| Probe cells (cheap first pass) | `(64 tok, ½ audio)`, `(64, 0)`, `(0, ½)` | below | Arjun, Lindy, Alvin |
| Rest of the budget grid | image {0, 64, 256, 768} tokens × audio {0, ¼, ½, 1}. The corners and text cells are not re-run, because they are the base formats. | below | Alvin |
| Audio selection | ¼ and ½ of the clip: centered (`c`) and loudest window (`e`), audio only; loudest also with a 64-token image | below, `src/idea3/cells.py` | Alvin |
| Idea 4 subset | 242 items: 120 midterm test items + 60 fresh speech + 60 fresh sound + 2 fresh music | `data/final_idea4_ids.json` | Nikkita |
| Speech vs non-speech | AST P(Speech) ≥ 0.5; any other threshold can be computed afterwards from §9c | `src/final/idea4_generate.py` | Nikkita |
| Audio tags | AST (AudioSet), top 5 with p ≥ 0.05 | same | Nikkita |
| Generation | greedy, ≤ 256 new tokens (the main setting, as at the midterm), stops at end of turn | same, `idea4_gen_lens` in `configs/default.yaml` | Nikkita |
| Generation-length ablation | 80, 120, 256 tokens. The 80- and 120-token texts are the first 80 / 120 tokens of the 256-token output: decoding is greedy, so that is exactly what a lower limit produces, and the time each limit takes is logged. Scored for the 6 configs that use Qwen-generated text; 256 is §9 itself | `idea4_gen_lens`; §9b | Nikkita |
| Question-conditioned prompt | the question only, never the options | same (`Q_AUDIO_PROMPT`) | Nikkita |
| ASR "broken" | empty / known filler / repetition loop → fall back to raw audio | `src/midterm/idea4_surrogates.py::asr_status` | Nikkita |
| Masking test | every option word (and its singular form) masked in the human description | `idea4_generate.py::mask_options` | Nikkita |

In [ ]:
# --- §0 run settings (shared by every section below) ---
PROBE_CELLS  = 'v64_a0.5 v64_a0 v0_a0.5'
OTHER_CELLS  = 'v0_a0.25 v64_a0.25 v64_a1 v256_a0 v256_a0.25 v256_a0.5 v256_a1 v768_a0.25 v768_a0.5'
SELECT_CELLS = 'v0_a0.25c v0_a0.5c v0_a0.25e v0_a0.5e v64_a0.25e v64_a0.5e'
IDEA4_CONFIGS = ('img_route img_route_fb img_qdesc img_pipe gencap_pipe img_masked cap_masked '  # new pipeline + ablations
                 'img_asr img_gendesc gen_cap_desc gen_cap_asr')                                # midterm baselines
IDEA4_LEN_BASES = 'img_qdesc img_pipe gencap_pipe img_gendesc gen_cap_desc gen_cap_asr'  # configs with Qwen-generated text
ABLATION_LENS = (80, 120)  # generation-length ablation; 256 is the §9 run. Must be in idea4_gen_lens (configs/default.yaml)
IDEA4_LEN_CONFIGS = ' '.join(f'{c}_t{n}' for n in ABLATION_LENS for c in IDEA4_LEN_BASES.split())
IDEA4_THR_CONFIGS = 'img_tags img_tagsq gencap_tagsq gencap_aud'  # routing branches on every item (speech-threshold sweep)
N_ITEMS, N_IDEA4, N_ORDERS = 1141, 242, 3
EXPECTED = {  # passes per output, for the progress check
    '06 base bank':        N_ITEMS * 7 * N_ORDERS,
    '07 probe cells':      N_ITEMS * len(PROBE_CELLS.split()) * N_ORDERS,
    '08 idea4 text':       N_IDEA4,  # items, not passes
    '09 idea4 scoring':    N_IDEA4 * len(IDEA4_CONFIGS.split()) * N_ORDERS,
    '09b idea4 lengths':   N_IDEA4 * len(IDEA4_LEN_CONFIGS.split()) * N_ORDERS,
    '09c idea4 threshold': N_IDEA4 * len(IDEA4_THR_CONFIGS.split()) * N_ORDERS,
    '10 idea3 grid':       N_ITEMS * len(OTHER_CELLS.split()) * N_ORDERS,
    '11 audio selection':  N_ITEMS * len(SELECT_CELLS.split()) * N_ORDERS,
}
print({k: f'{v:,}' for k, v in EXPECTED.items()}, '| total passes', f"{sum(v for k, v in EXPECTED.items() if 'text' not in k):,}")

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Google Drive (where all outputs go)

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/mmml_final'
for sub in ('logs', 'backups'):
    os.makedirs(f'{DRIVE}/{sub}', exist_ok=True)

# Optional: keep the model and dataset downloads on Drive, so a reconnect skips re-downloading them.
# This needs roughly 15 GB of free Drive space. Leave it False if Drive is tight; a fresh download takes a few minutes.
CACHE_ON_DRIVE = False
if CACHE_ON_DRIVE:
    os.environ['HF_HOME'] = f'{DRIVE}/hf_cache'
print(sorted(os.listdir(DRIVE)))

## 2. Get the code

If the repo is private, first add a GitHub token as a Colab secret named `GH_TOKEN` (key icon in the left sidebar).
The commit printed at the end is the exact code version of this run: note it in the report's reproducibility details.

In [ ]:
REPO = '/content/multimodal-reasoning-project'
URL = 'https://github.com/alvinwang1/multimodal-reasoning-project.git'
try:
    from google.colab import userdata
    URL = URL.replace('https://', f"https://{userdata.get('GH_TOKEN')}@")
except Exception:
    pass  # public repo, or no token saved
if not os.path.exists(REPO):
    !git clone -q {URL} {REPO}
else:
    !git -C {REPO} pull -q
%cd {REPO}
!git log --oneline -1 | tee -a {DRIVE}/logs/00_code_version.log

## 3. Install dependencies

This keeps Colab's own CUDA build of PyTorch and pins the model and audio libraries to the versions used on the Mac.

In [ ]:
!pip install -q transformers==5.18.0 accelerate==1.15.0 qwen-omni-utils==0.0.9 datasets==5.0.1 \
    librosa==1.0.0 av==19.0.1 soundfile pyyaml scikit-learn sentence-transformers
import torch, transformers
print('torch', torch.__version__, '| transformers', transformers.__version__, '| cuda', torch.cuda.is_available())

## 4. Outputs → Drive, backups, progress

The runners append one line per pass to files that live in Drive, through symlinks. The Idea 4 text file is written straight into Drive.
Run `progress()` in any cell, at any time, to see how far each section has got.

In [ ]:
import glob, json, shutil, threading, time
from collections import Counter
from datetime import datetime

LINKS = {
    'results/final/smoke_l4.jsonl': 'smoke_l4.jsonl',
    'results/final/bank_base_l4.jsonl': 'bank_base_l4.jsonl',
    'results/idea3/raw_final.jsonl': 'idea3_raw_final.jsonl',
    'results/final/raw_idea4_l4.jsonl': 'raw_idea4_l4.jsonl',
}
for rel, name in LINKS.items():
    target = f'{DRIVE}/{name}'
    open(target, 'a').close()
    if os.path.islink(rel):
        os.remove(rel)
    elif os.path.exists(rel):
        raise RuntimeError(f'{rel} is a real file; move it into {DRIVE} first so nothing is lost')
    os.makedirs(os.path.dirname(rel), exist_ok=True)
    os.symlink(target, rel)
IDEA4_TEXT = f'{DRIVE}/idea4_surrogates.json'
FILTER = 'UNEXPECTED|^token2wav|^talker|Model config|^Key |^-+\\+|^Notes:|can be ignored|LOAD REPORT|^\\s*$|Loading weights'

# ---- backups: timestamped copies of every output in Drive/backups, now and every 15 min (keep the last 3)
BACKUP_EVERY_MIN, KEEP = 15, 3
OUTPUTS = [f'{DRIVE}/{name}' for name in LINKS.values()] + [IDEA4_TEXT]

def backup(tag=None):
    stamp = tag or datetime.now().strftime('%m%d-%H%M')
    for path in OUTPUTS:
        if os.path.exists(path) and os.path.getsize(path):
            name = os.path.basename(path)
            shutil.copy2(path, f'{DRIVE}/backups/{name}.{stamp}')
            for old in sorted(glob.glob(f'{DRIVE}/backups/{name}.*'), key=os.path.getmtime)[:-KEEP]:
                os.remove(old)

def _backup_loop():
    while True:
        time.sleep(BACKUP_EVERY_MIN * 60)
        try:
            backup()
        except Exception as e:  # never let a backup hiccup stop the runs
            print('backup failed:', e)

if '_backup_thread' not in globals():
    _backup_thread = threading.Thread(target=_backup_loop, daemon=True)
    _backup_thread.start()
backup(f"start-{datetime.now():%m%d-%H%M}")

# ---- progress: passes done per section (unique item x config x order; cut-off lines ignored)
def _keys(path, key):
    done = set()
    if os.path.exists(path):
        for line in open(path):
            try:
                r = json.loads(line)
                done.add((r['id'], r[key], r['perm']))
            except (json.JSONDecodeError, KeyError):
                pass
    return done

def progress():
    base = _keys(f'{DRIVE}/bank_base_l4.jsonl', 'config')
    i3 = Counter(c for _, c, _ in _keys(f'{DRIVE}/idea3_raw_final.jsonl', 'cell'))
    i4 = Counter(c for _, c, _ in _keys(f'{DRIVE}/raw_idea4_l4.jsonl', 'config'))
    text = len(json.load(open(IDEA4_TEXT))) if os.path.exists(IDEA4_TEXT) and os.path.getsize(IDEA4_TEXT) else 0
    done = {'06 base bank': len(base),
            '07 probe cells': sum(i3[c] for c in PROBE_CELLS.split()),
            '08 idea4 text': text,
            '09 idea4 scoring': sum(i4[c] for c in IDEA4_CONFIGS.split()),
            '09b idea4 lengths': sum(i4[c] for c in IDEA4_LEN_CONFIGS.split()),
            '09c idea4 threshold': sum(i4[c] for c in IDEA4_THR_CONFIGS.split()),
            '10 idea3 grid': sum(i3[c] for c in OTHER_CELLS.split()),
            '11 audio selection': sum(i3[c] for c in SELECT_CELLS.split())}
    for k, n in done.items():
        print(f'{k:20s} {n:>7,} / {EXPECTED[k]:>7,}  {100 * n / EXPECTED[k]:5.1f}%')

progress()

## 5. Smoke test (about 5 min, mostly the model download)

Check the log for `model on cuda (torch.bfloat16)`, `top_vocab_is_letter` and the warmup times. After warmup, `full` should take under 1 s per pass.

In [ ]:
!python -m src.analysis_a.run --split final --limit 2 --configs full cap_trans --out results/final/smoke_l4.jsonl 2>&1 | grep -vE '{FILTER}' | tee -a {DRIVE}/logs/05_smoke.log

## 6. Base bank: Ideas 1, 3, 4 (~1.6 h) · Arjun, Lindy

The 7 base input formats × 3 option orders on all 1,141 items: full, image+transcript, caption+audio, caption+transcript, image only, audio only, question only.
These are every idea's baselines, and the labels for the Idea 1 router.

In [ ]:
!python -m src.analysis_a.run --split final --perm-configs all --out results/final/bank_base_l4.jsonl 2>&1 | grep --line-buffered -vE '{FILTER}' | tee -a {DRIVE}/logs/06_base_bank.log

## 7. Cheap probe cells: Ideas 1 and 3 (~0.7 h) · Arjun, Lindy, Alvin

A 64-token image with half the audio, a 64-token image alone, and half the audio alone.
For Idea 1 they give the disagreement features (KL between the image-only and audio-only answers); for Idea 3 they are the cheap first pass.

In [ ]:
!python -m src.idea3.run --split final --cells {PROBE_CELLS} 2>&1 | grep --line-buffered -vE '{FILTER}' | tee -a {DRIVE}/logs/07_probe_cells.log

## 8. Idea 4: generate the text substitutes (~55 min) · Nikkita

For each of the 242 items:
- Whisper transcript;
- AST audio tags and P(speech);
- Qwen's generic caption and audio description (the midterm baseline);
- Qwen's **question-conditioned** audio description;
- the human description with the option words masked.

Each step's time is logged, for the generation-cost accounting.

Qwen writes each text once, with at most 256 new tokens. The 80- and 120-token versions for §9b are the first 80 and 120 tokens of that same output (`<field>_t80`, `<field>_t120`). Decoding is greedy, so this is exactly what an 80- or 120-token limit would produce, at no extra GPU cost; the time each limit would take is logged as `<field>_t<n>_ms`.

In [ ]:
!python -m src.final.idea4_generate --out {IDEA4_TEXT} 2>&1 | grep --line-buffered -vE '{FILTER}|warn|^\[transformers\]' | tee -a {DRIVE}/logs/08_idea4_generate.log

## 9. Idea 4: score the configs (~35 min) · Nikkita

**New pipeline and ablations:**
- `img_route`: speech → Whisper, other audio → AST tags;
- `img_route_fb`: the same, plus a fallback to raw audio when the transcript is broken;
- `img_qdesc`: the question-conditioned description;
- `img_pipe` / `gencap_pipe`: the full pipeline, with the image or with the generated caption;
- `img_masked` / `cap_masked`: the masking test.

**Midterm baselines, regenerated on this subset:** `img_asr`, `img_gendesc`, `gen_cap_desc`, `gen_cap_asr`.

The raw and reference-text formats for these items are already in the §6 bank.

In [ ]:
!python -m src.analysis_a.run --split idea4 --configs {IDEA4_CONFIGS} --perm-configs all --surrogates {IDEA4_TEXT} --out results/final/raw_idea4_l4.jsonl 2>&1 | grep --line-buffered -vE '{FILTER}' | tee -a {DRIVE}/logs/09_idea4_score.log

## 9b. Idea 4: generation-length ablation (~30 min) · Nikkita

The 6 Idea 4 configs whose text Qwen generates (`img_qdesc`, `img_pipe`, `gencap_pipe`, `img_gendesc`, `gen_cap_desc`, `gen_cap_asr`), with that text capped at **80** and **120** new tokens (`<config>_t80`, `<config>_t120`). The **256**-token point is §9. The capped texts come from §8, so §8 must have finished all 242 items; the runner checks this before it loads the model.

`img_pipe` gives speech-routed items the Whisper transcript, which the cap does not change, so it only differs on non-speech items.

In [ ]:
!python -m src.analysis_a.run --split idea4 --configs {IDEA4_LEN_CONFIGS} --perm-configs all --surrogates {IDEA4_TEXT} --out results/final/raw_idea4_l4.jsonl 2>&1 | grep --line-buffered -vE '{FILTER}' | tee -a {DRIVE}/logs/09b_idea4_lengths.log

## 9c. Idea 4: inputs for the speech-threshold sweep (~12 min) · Nikkita

§9 routes each clip once, at AST P(Speech) ≥ 0.5, so each item is only scored on the branch it was routed to. Here every item is also scored on the non-speech branches. Any threshold τ can then be computed afterwards from the bank, with no more GPU time, by picking each item's branch:

| Routed config | P(Speech) ≥ τ | P(Speech) < τ |
|---|---|---|
| `img_route` | `img_asr` | `img_tags` |
| `img_route_fb` | `img_asr`, or `full` (§6) if the transcript is broken | `img_tags` |
| `img_pipe` | `img_asr`, or `full` if broken | `img_tagsq` |
| `gencap_pipe` | `gen_cap_asr`, or `gencap_aud` if broken | `gencap_tagsq` |

P(Speech) is `ast_p_speech` and "broken" is `asr_status != "ok"`, both in `idea4_surrogates.json`. At τ = 0.5 the table gives exactly the prompts §9 scored. The fallback detector can be varied the same way, since `img_asr` and `full` are scored on every item.

In [ ]:
!python -m src.analysis_a.run --split idea4 --configs {IDEA4_THR_CONFIGS} --perm-configs all --surrogates {IDEA4_TEXT} --out results/final/raw_idea4_l4.jsonl 2>&1 | grep --line-buffered -vE '{FILTER}' | tee -a {DRIVE}/logs/09c_idea4_threshold.log

## 10. Idea 3: the other 9 budget-grid cells (~2 h) · Alvin

`v<image tokens>_a<audio fraction>`, keeping the **first** part of the clip. Together with §6 (corners and text cells) and §7 this completes the 4×4 grid.

In [ ]:
!python -m src.idea3.run --split final --cells {OTHER_CELLS} 2>&1 | grep --line-buffered -vE '{FILTER}' | tee -a {DRIVE}/logs/10_idea3_grid.log

## 11. Idea 3: which part of the audio to keep (~1 h) · Alvin

The same budgets as §10's first-part cells, but keeping the **centered** window (`c`) or the **loudest** window (`e`: the contiguous window with the most signal energy).
Audio only isolates the effect; the 64-token-image versions match the allocator's menu.

In [ ]:
!python -m src.idea3.run --split final --cells {SELECT_CELLS} 2>&1 | grep --line-buffered -vE '{FILTER}' | tee -a {DRIVE}/logs/11_idea3_audio_select.log

## 12. Final check and hand-off

- `progress()` should show 100% everywhere.
- The bank should show every base format and `i3:` cell at 1,141 items, and every `i4:` config at 242.
- Share `MyDrive/mmml_final/` with the team. Each person then downloads these files into the same repo paths on the Mac (they are gitignored):
  - `bank_base_l4.jsonl` → `results/final/bank_base_l4.jsonl`
  - `idea3_raw_final.jsonl` → `results/idea3/raw_final.jsonl`
  - `raw_idea4_l4.jsonl` → `results/final/raw_idea4_l4.jsonl`
  - `idea4_surrogates.json` → `results/final/idea4_surrogates.json`

**If a results file ever looks wrong** (e.g. the progress drops after a reconnect), copy the newest good snapshot from `backups/` over it and rerun that section.

In [ ]:
progress()
!cp {IDEA4_TEXT} results/final/idea4_surrogates.json
!python -m src.final.bank | tee -a {DRIVE}/logs/12_coverage.log
backup('final')
print('final backup written to', f'{DRIVE}/backups')